In [1]:
import numpy as np
import sys
import os
import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa
from utils.bam import MultiBAMv4

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\my_lora_utils.py


In [2]:
sf = 9
bw = 125000
fs = 1000000
lora_init = LoRa(sf, bw)
input_row = 512
input_col = 33
second_layer = 2048
third_layer = 256

output_classifier = int(2 ** sf)

## HOW TO LOAD WEIGHT
layers = [input_col * input_row, second_layer, third_layer] # <-- must match training

multi_bam = MultiBAMv4(layers_dims=layers, eta=1e-5)

for i, bam in enumerate(multi_bam.bams):
    path = f"weight_{input_row*input_col}_{second_layer}_{third_layer}/weights_layer_{i}.npy"

    if os.path.isfile(path):
        w_np = np.load(path)
        bam.W = torch.tensor(w_np, dtype=torch.float32, device=bam.device)
        print(f"✅ Loaded layer {i} | shape {bam.W.shape}")
    else:
        print(f"⚠️ Missing file for layer {i}: {path}")


✅ Loaded layer 0 | shape torch.Size([2048, 16896])
✅ Loaded layer 1 | shape torch.Size([256, 2048])


In [3]:
GEENRATE_ = True
SNR_first = 30
SNR_last = -30
seed = None

if (GEENRATE_):

    folder_path = f'classifier_dataset_sf{sf}_{third_layer}_{output_classifier}'
    X_data = []
    y_data = []
    # Check if folder exists, if not create it
    check_and_make_folder(folder_path)

    # Outer loop: from 15 to -30 (inclusive)
    for i in range(SNR_first, SNR_last, -1):  # step -1 to count down
        # Inner loop: 100 iteration

        for j in range(output_classifier): # 0 until 2**sf
            x1 = lora_init.gen_symbol_fs(j, sf=sf, bw=bw, Fs=fs)  # you had Fs=int(bw*8)=1e6
            x = lora_init.awgn_iq_with_seed(x1,i,seed)
            input_spec = create_spectrogram_from_torch(x,sf,bw,fs,input_row,input_col,0,0,1,None)
            flat = input_spec.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
            out1 = multi_bam.compress(flat)
            out = out1.squeeze() 
            X_data.append(out)
            y_data.append(j)
    
    X_data = np.stack(X_data)   # (N, 16)
    y_data = np.array(y_data)
    print(X_data.shape)
    print(y_data.shape)
    np.save(f"{folder_path}/X.npy", X_data)
    np.save(f"{folder_path}/y.npy", y_data)



Folder created: classifier_dataset_sf9_256_512


c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\bam.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32, device=self.device)


(30720, 256)
(30720,)
